In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. 文件路径配置 ---
glassdoor_panel_file = 'firm_monthly_panel_data_revised.csv'
mapping_file = 'glassdoor_to_wrds_mapping.csv' # 使用未经筛选的原始匹配文件
financial_panel_file = 'financial_panel_data_compustat_only.csv'
final_output_file = 'final_master_panel_for_analysis_strict.csv'

# --- 2. 加载所有数据文件 ---
print("--- 正在加载所有阶段的数据文件... ---")
try:
    df_glassdoor = pd.read_csv(glassdoor_panel_file)
    df_raw_mapping = pd.read_csv(mapping_file)
    df_financial = pd.read_csv(financial_panel_file)
    print("所有数据文件加载成功。")
except FileNotFoundError as e:
    print(f"错误：文件未找到 -> {e}")
    exit()

# --- 3. 筛选匹配结果 (与之前相同) ---

SIMILARITY_THRESHOLD = 0.95 
print(f"\n--- 应用相似度阈值: >= {SIMILARITY_THRESHOLD} ---")
df_filtered_mapping = df_raw_mapping[df_raw_mapping['similarity'] >= SIMILARITY_THRESHOLD].copy()
print(f"原始潜在匹配数: {len(df_raw_mapping)}")
print(f"筛选后高置信度匹配数: {len(df_filtered_mapping)}")

# --- 4. 准备数据并初步合并 ---
print("\n--- 正在统一日期格式并执行初步合并... ---")
df_glassdoor['year_month'] = pd.to_datetime(df_glassdoor['year_month']).dt.to_period('M')
df_financial['year_month'] = pd.to_datetime(df_financial['year_month']).dt.to_period('M')

# 合并 Glassdoor + Mapping + Financial
df_merged = pd.merge(df_glassdoor, df_filtered_mapping[['company_code', 'gvkey']], on='company_code', how='inner')
df_panel = pd.merge(df_merged, df_financial, on=['gvkey', 'year_month'], how='left')
print(f"初步合并完成，当前数据集包含 {len(df_panel)} 行。")

# ==============================================================================
# --- 5. 执行严格的数据筛选流程  ---
# ==============================================================================
print("\n" + "="*60)
print("     开始执行严格的数据筛选流程 (复现原文 Table A.2)")
print("="*60)

# 步骤 A: 时间窗口筛选 (Timeline Filter)
print(f"\n步骤 A: 应用时间窗口 (Nov 2017 - Apr 2019)...")
start_date = pd.Period('2017-11', 'M')
end_date = pd.Period('2019-04', 'M')
df_panel_filtered = df_panel[(df_panel['year_month'] >= start_date) & (df_panel['year_month'] <= end_date)].copy()
print(f"筛选后剩余行数: {len(df_panel_filtered)}")

# 步骤 B: 控制变量缺失值过滤 (Missing Controls Filter)
print(f"\n步骤 B: 剔除关键财务变量全部缺失的行...")
control_vars = ['log_assets', 'intangible_asset_ratio', 'book_to_market', 'earnings_surprise', 'cash_holding']

initial_rows = len(df_panel_filtered)
df_panel_filtered.dropna(subset=control_vars, how='all', inplace=True)
print(f"剔除了 {initial_rows - len(df_panel_filtered)} 行。筛选后剩余行数: {len(df_panel_filtered)}")

# 步骤 C: 核心因果推断样本筛选 (Pre-Post Requirement Filter)
print(f"\n步骤 C: 核心筛选 - 只保留在事件前后都有评级的公司...")
# 定义伪事件窗口 (基于原文Table A.1)
pseudo_event_month = pd.Period('2018-04', 'M')
pre_period_start = pd.Period('2018-01', 'M')
pre_period_end = pd.Period('2018-03', 'M')
post_period_start = pd.Period('2018-04', 'M')
post_period_end = pd.Period('2018-06', 'M')
print(f"伪事件月份: {pseudo_event_month}, 前期: {pre_period_start}-{pre_period_end}, 后期: {post_period_start}-{post_period_end}")

# 标记每个公司是否在前期和后期都有有效评级
df_panel_filtered['has_pre_rating'] = (
    df_panel_filtered['compensation_benefits_mean'].notna() &
    (df_panel_filtered['year_month'] >= pre_period_start) &
    (df_panel_filtered['year_month'] <= pre_period_end)
)
df_panel_filtered['has_post_rating'] = (
    df_panel_filtered['compensation_benefits_mean'].notna() &
    (df_panel_filtered['year_month'] >= post_period_start) &
    (df_panel_filtered['year_month'] <= post_period_end)
)

# 按公司分组，判断是否同时满足前期和后期条件
firm_validity = df_panel_filtered.groupby('gvkey').agg(
    has_any_pre_rating=('has_pre_rating', 'any'),
    has_any_post_rating=('has_post_rating', 'any')
)
firms_to_keep = firm_validity[firm_validity['has_any_pre_rating'] & firm_validity['has_any_post_rating']].index
print(f"共有 {len(firms_to_keep)} 家公司满足事件前后都有评级的条件。")


initial_firm_count = df_panel_filtered['gvkey'].nunique()
df_panel_filtered = df_panel_filtered[df_panel_filtered['gvkey'].isin(firms_to_keep)].copy()
print(f"公司数量从 {initial_firm_count} 减少到 {df_panel_filtered['gvkey'].nunique()}。")
print(f"筛选后剩余行数: {len(df_panel_filtered)}")

# 步骤 D: 因变量缺失值过滤 
print(f"\n步骤 D: 剔除因变量 (Compensation-and-Benefits Rating) 缺失的最终行...")
initial_rows = len(df_panel_filtered)
df_panel_filtered.dropna(subset=['compensation_benefits_mean'], inplace=True)
print(f"剔除了 {initial_rows - len(df_panel_filtered)} 行。最终样本行数: {len(df_panel_filtered)}")
print("="*60)
print("     严格筛选流程完成！")
print("="*60)


# --- 6. 保存并生成最终的描述性统计表 ---
df_final_panel = df_panel_filtered.copy()
df_final_panel.to_csv(final_output_file, index=False, encoding='utf-8-sig')
print(f"\n--- 成功！经过严格筛选的最终面板数据已保存到 '{final_output_file}' ---")

# (生成统计表的代码与之前版本相同)
print("\n--- 正在生成最终的描述性统计表... ---")
rename_map_for_table1 = {
    'monthly_rating_count': 'Raw Number of Ratings Each Month', 'compensation_benefits_mean': 'Compensation-and-Benefits Rating',
    'work_life_balance_mean': 'Work-Life-Balance Rating', 'compensation_benefits_dispersion': 'Compensation-and-Benefits Dispersion',
    'log_assets': 'Log Assets', 'intangible_asset_ratio': 'Intangible Asset', 'book_to_market': 'Book to Market',
    'earnings_surprise': 'Earnings Surprise', 'cash_holding': 'Cash Holding'
}
df_for_stats = df_final_panel[rename_map_for_table1.keys()].rename(columns=rename_map_for_table1)
desc_stats = df_for_stats.describe(percentiles=[.25, .5, .75]).transpose()
desc_stats = desc_stats.rename(columns={'count': 'Observations', 'mean': 'Mean', 'std': 'St. Dev.', '25%': 'P25', '50%': 'P50', '75%': 'P75'})
final_table = desc_stats[['Observations', 'Mean', 'St. Dev.', 'P25', 'P50', 'P75']]
final_table['Observations'] = final_table['Observations'].astype(int)

print("\n" + "="*50)
print("     复现的描述性统计表 (基于严格筛选的样本)")
print("="*50)
print(final_table)
print("="*50)

--- 正在加载所有阶段的数据文件... ---
所有数据文件加载成功。

--- 应用相似度阈值: >= 0.95 ---
原始潜在匹配数: 26114
筛选后高置信度匹配数: 4533

--- 正在统一日期格式并执行初步合并... ---
初步合并完成，当前数据集包含 269635 行。

     开始执行严格的数据筛选流程 (复现原文 Table A.2)

步骤 A: 应用时间窗口 (Nov 2017 - Apr 2019)...
筛选后剩余行数: 32062

步骤 B: 剔除关键财务变量全部缺失的行...
剔除了 14900 行。筛选后剩余行数: 17162

步骤 C: 核心筛选 - 只保留在事件前后都有评级的公司...
伪事件月份: 2018-04, 前期: 2018-01-2018-03, 后期: 2018-04-2018-06
共有 942 家公司满足事件前后都有评级的条件。
公司数量从 1366 减少到 942。
筛选后剩余行数: 15632

步骤 D: 剔除因变量 (Compensation-and-Benefits Rating) 缺失的最终行...
剔除了 335 行。最终样本行数: 15297
     严格筛选流程完成！

--- 成功！经过严格筛选的最终面板数据已保存到 'final_master_panel_for_analysis_strict.csv' ---

--- 正在生成最终的描述性统计表... ---

     复现的描述性统计表 (基于严格筛选的样本)
                                      Observations      Mean   St. Dev.  \
Raw Number of Ratings Each Month             15297  0.048245   1.414223   
Compensation-and-Benefits Rating             15297  3.360962   0.849329   
Work-Life-Balance Rating                     15290  3.282143   0.912774   
Compensation-and-Benefits Dispers

C:\Users\32678\AppData\Local\Temp\ipykernel_9032\3311162534.py:126: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table['Observations'] = final_table['Observations'].astype(int)


In [18]:
import pandas as pd

# --- 1. 文件路径配置 ---
# 确保这个文件是您在第四阶段(严格清洗版)中生成的最终面板
final_panel_file = 'final_master_panel_for_analysis_strict.csv'

# --- 2. 加载最终面板数据 ---
print("--- 正在加载经过严格筛选的主面板数据... ---")
try:
    df_final_panel = pd.read_csv(final_panel_file)
    print(f"文件 '{final_panel_file}' 加载成功。")
except FileNotFoundError:
    print(f"错误：找不到文件 '{final_panel_file}'。")
    print("请确保您已成功运行了第四阶段（严格清洗版）的代码。")
    exit()

# --- 3. 准备用于生成统计表的数据 ---
# 解释：这部分与第四阶段的最后一步相同，我们重新计算统计数据以确保一致性。
print("--- 正在计算描述性统计... ---")

# 定义要包含在表中的变量，并重命名以匹配论文
rename_map_for_table1 = {
    'monthly_rating_count': 'Raw Number of Ratings Each Month',
    'compensation_benefits_mean': 'Compensation-and-Benefits Rating',
    'work_life_balance_mean': 'Work-Life-Balance Rating',
    'compensation_benefits_dispersion': 'Compensation-and-Benefits Dispersion',
    'log_assets': 'Log Assets',
    'intangible_asset_ratio': 'Intangible Asset',
    'book_to_market': 'Book to Market',
    'earnings_surprise': 'Earnings Surprise',
    'cash_holding': 'Cash Holding'
}
df_for_stats = df_final_panel[rename_map_for_table1.keys()].rename(columns=rename_map_for_table1)

# 计算描述性统计
desc_stats = df_for_stats.describe(percentiles=[.25, .5, .75]).transpose()

# 格式化DataFrame以匹配论文Table 1
desc_stats = desc_stats.rename(columns={
    'count': 'Observations', 'mean': 'Mean', 'std': 'St. Dev.',
    '25%': 'P25', '50%': 'P50', '75%': 'P75'
})
final_table = desc_stats[['Observations', 'Mean', 'St. Dev.', 'P25', 'P50', 'P75']]
final_table['Observations'] = final_table['Observations'].astype(int)

# --- 4. 关键步骤：生成LaTeX代码 ---
# 解释：我们使用 Pandas 的 .to_latex() 方法。
# 如果您的 Pandas 版本较旧，请参考我们之前的讨论，使用兼容性方案B。
# 这里假设您已升级 Pandas 或使用的是较新版本。
print("\n--- 正在生成LaTeX代码... ---")
try:
    latex_code = final_table.to_latex(
        index=True,
        header=True,
        float_format="%.2f",
        booktabs=True,
        caption="Descriptive Statistics (Based on Strictly Filtered Sample)",
        label="tab:desc_stats_strict",
        column_format="lrrrrrr" # l: 变量名列左对齐, 6*r: 6个数值列右对齐
    )
except TypeError as e:
    print("\n警告：您的Pandas版本较旧，不支持'booktabs'等参数。正在切换到兼容模式...")
    print(f"错误信息: {e}")
    
    # 方案B：兼容旧版Pandas的语法
    core_latex_table = final_table.to_latex(
        index=True, header=True, float_format="%.2f", column_format="lrrrrrr"
    )
    caption_text = "Descriptive Statistics (Based on Strictly Filtered Sample)"
    label_text = "tab:desc_stats_strict"
    latex_code = f"""
\\begin{{table}}[htbp]
\\centering
\\caption{{{caption_text}}}
\\label{{{label_text}}}
{core_latex_table}
\\end{{table}}
"""

# --- 5. 打印最终的LaTeX代码 ---
print("\n" + "="*70)
print("     将以下代码复制并粘贴到您的 .tex 文件中")
print("="*70)
print(latex_code)
print("="*70)


--- 正在加载经过严格筛选的主面板数据... ---
文件 'final_master_panel_for_analysis_strict.csv' 加载成功。
--- 正在计算描述性统计... ---

--- 正在生成LaTeX代码... ---

警告：您的Pandas版本较旧，不支持'booktabs'等参数。正在切换到兼容模式...
错误信息: NDFrame.to_latex() got an unexpected keyword argument 'booktabs'

     将以下代码复制并粘贴到您的 .tex 文件中

\begin{table}[htbp]
\centering
\caption{Descriptive Statistics (Based on Strictly Filtered Sample)}
\label{tab:desc_stats_strict}
\begin{tabular}{lrrrrrr}
\toprule
 & Observations & Mean & St. Dev. & P25 & P50 & P75 \\
\midrule
Raw Number of Ratings Each Month & 15297 & 0.05 & 1.41 & 0.00 & 0.00 & 0.00 \\
Compensation-and-Benefits Rating & 15297 & 3.36 & 0.85 & 2.95 & 3.41 & 4.00 \\
Work-Life-Balance Rating & 15290 & 3.28 & 0.91 & 2.81 & 3.33 & 4.00 \\
Compensation-and-Benefits Dispersion & 12367 & 1.08 & 0.46 & 0.82 & 1.11 & 1.34 \\
Log Assets & 15294 & 8.60 & 2.21 & 7.28 & 8.62 & 9.97 \\
Intangible Asset & 15263 & 0.24 & 0.22 & 0.04 & 0.18 & 0.39 \\
Book to Market & 14862 & -1.04 & 23.98 & 0.18 & 0.34 & 0.61 \\
Ea

C:\Users\32678\AppData\Local\Temp\ipykernel_9032\2169728296.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table['Observations'] = final_table['Observations'].astype(int)


In [19]:
import pandas as pd

# --- 1. 加载和准备数据 (与之前相同) ---
glassdoor_panel_file = 'firm_monthly_panel_data_revised.csv'
mapping_file = 'glassdoor_to_wrds_mapping.csv'
financial_panel_file = 'financial_panel_data_compustat_only.csv'

df_glassdoor = pd.read_csv(glassdoor_panel_file)
df_raw_mapping = pd.read_csv(mapping_file)
df_financial = pd.read_csv(financial_panel_file)

SIMILARITY_THRESHOLD = 0.90 
df_filtered_mapping = df_raw_mapping[df_raw_mapping['similarity'] >= SIMILARITY_THRESHOLD].copy()

df_glassdoor['year_month'] = pd.to_datetime(df_glassdoor['year_month']).dt.to_period('M')
df_financial['year_month'] = pd.to_datetime(df_financial['year_month']).dt.to_period('M')

df_merged = pd.merge(df_glassdoor, df_filtered_mapping[['company_code', 'gvkey']], on='company_code', how='inner')
df_panel = pd.merge(df_merged, df_financial, on=['gvkey', 'year_month'], how='left')


# --- 2. 执行严格的数据筛选流程 (与之前相同) ---
# (为了简洁，这里只展示核心逻辑，不重复打印信息)
start_date = pd.Period('2017-11', 'M')
end_date = pd.Period('2019-04', 'M')
df_panel_filtered = df_panel[(df_panel['year_month'] >= start_date) & (df_panel['year_month'] <= end_date)].copy()

control_vars = ['log_assets', 'intangible_asset_ratio', 'book_to_market', 'earnings_surprise', 'cash_holding']
df_panel_filtered.dropna(subset=control_vars, how='all', inplace=True)

pseudo_event_month = pd.Period('2018-04', 'M')
pre_period_start = pd.Period('2018-01', 'M'); pre_period_end = pd.Period('2018-03', 'M')
post_period_start = pd.Period('2018-04', 'M'); post_period_end = pd.Period('2018-06', 'M')

df_panel_filtered['has_pre_rating'] = (df_panel_filtered['compensation_benefits_mean'].notna() & (df_panel_filtered['year_month'] >= pre_period_start) & (df_panel_filtered['year_month'] <= pre_period_end))
df_panel_filtered['has_post_rating'] = (df_panel_filtered['compensation_benefits_mean'].notna() & (df_panel_filtered['year_month'] >= post_period_start) & (df_panel_filtered['year_month'] <= post_period_end))
firm_validity = df_panel_filtered.groupby('gvkey').agg(has_any_pre_rating=('has_pre_rating', 'any'), has_any_post_rating=('has_post_rating', 'any'))
firms_to_keep = firm_validity[firm_validity['has_any_pre_rating'] & firm_validity['has_any_post_rating']].index

# ==============================================================================
# --- 3. 关键修改：创建两个不同阶段的样本 ---
# ==============================================================================

# 样本1: "描述性样本" (Descriptive Sample)
# 这是最终入选公司在筛选掉因变量缺失之前的样本，对应原文的 22,565 行
df_descriptive_sample = df_panel_filtered[df_panel_filtered['gvkey'].isin(firms_to_keep)].copy()

# 样本2: "回归样本" (Regression Sample)
# 这是最终用于回归的样本，对应原文的 18,647 行
df_regression_sample = df_descriptive_sample.dropna(subset=['compensation_benefits_mean']).copy()

print("\n--- 样本构建完成 ---")
print(f"最终入选公司数量: {len(firms_to_keep)}")
print(f"描述性样本行数 (用于计算Rating Count等): {len(df_descriptive_sample)}")
print(f"最终回归样本行数 (用于计算Rating Mean等): {len(df_regression_sample)}")


# --- 4. 分别在不同样本上计算统计量 ---
print("\n--- 正在生成描述性统计表 (模拟原文逻辑)... ---")

# 定义重命名映射
rename_map = {
    'monthly_rating_count': 'Raw Number of Ratings Each Month', 'compensation_benefits_mean': 'Compensation-and-Benefits Rating',
    'work_life_balance_mean': 'Work-Life-Balance Rating', 'compensation_benefits_dispersion': 'Compensation-and-Benefits Dispersion',
    'log_assets': 'Log Assets', 'intangible_asset_ratio': 'Intangible Asset', 'book_to_market': 'Book to Market',
    'earnings_surprise': 'Earnings Surprise', 'cash_holding': 'Cash Holding'
}

# 在 "回归样本" 上计算大部分变量
stats_regression = df_regression_sample[rename_map.keys()].describe(percentiles=[.25, .5, .75]).transpose()

# **单独在 "描述性样本" 上计算 Raw Number of Ratings**
stats_rating_count = df_descriptive_sample[['monthly_rating_count']].describe(percentiles=[.25, .5, .75]).transpose()

# --- 5. 合并统计结果并格式化 ---
# 从回归样本的统计中移除 rating_count
stats_regression = stats_regression.drop(index=['monthly_rating_count'])

# 将单独计算的 rating_count 统计加回来
final_stats_df = pd.concat([stats_rating_count, stats_regression])

# 重命名和格式化
final_stats_df = final_stats_df.rename(index=rename_map)
final_stats_df = final_stats_df.rename(columns={'count': 'Observations', 'mean': 'Mean', 'std': 'St. Dev.', '25%': 'P25', '50%': 'P50', '75%': 'P75'})
final_table = final_stats_df[['Observations', 'Mean', 'St. Dev.', 'P25', 'P50', 'P75']]
final_table['Observations'] = final_table['Observations'].astype(int)

# 按照原文顺序排序
final_table = final_table.reindex(index=rename_map.values())


# --- 6. 打印最终结果 ---
print("\n" + "="*50)
print("     复现的描述性统计表 (模拟原文分步逻辑)")
print("="*50)
print(final_table)
print("="*50)


--- 样本构建完成 ---
最终入选公司数量: 972
描述性样本行数 (用于计算Rating Count等): 17215
最终回归样本行数 (用于计算Rating Mean等): 16804

--- 正在生成描述性统计表 (模拟原文逻辑)... ---

     复现的描述性统计表 (模拟原文分步逻辑)
                                      Observations      Mean   St. Dev.  \
Raw Number of Ratings Each Month             17215  0.042870   1.333196   
Compensation-and-Benefits Rating             16804  3.349742   0.863824   
Work-Life-Balance Rating                     16793  3.283327   0.923165   
Compensation-and-Benefits Dispersion         13361  1.075861   0.460935   
Log Assets                                   16801  8.400935   2.362502   
Intangible Asset                             16751  0.230725   0.218673   
Book to Market                               16293 -0.893123  22.909759   
Earnings Surprise                            16335  0.024544   2.348516   
Cash Holding                                 16801  0.150817   0.173315   

                                           P25       P50       P75  
Raw Number of Ratings

C:\Users\32678\AppData\Local\Temp\ipykernel_9032\2563606741.py:86: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_table['Observations'] = final_table['Observations'].astype(int)
